# Phase 3 — Feature Extraction & Dataset Splitting
## Analisis Sentimen Ulasan Pengguna Grand Theft Auto VI

Notebook ini mendokumentasikan tahapan pembagian dataset (Train-Test Split) dan ekstraksi fitur berbasis TF-IDF (Term Frequency-Inverse Document Frequency) tanpa terjadi data leakage.

### 1. Inisialisasi dan Pemuatan Dataset Bersih

In [ ]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Baca dataset hasil preprocessing
df = pd.read_csv('../data/processed/processed_reviews.csv')
print(f"Total data: {len(df):,} baris")
df.head()

### 2. Dataset Preparation & Exclusion of Non-Feature Columns
Kolom `score`, `review_id`, `platform`, dan `published_date` secara eksplisit dikecualikan dari fitur model agar terbebas dari data leakage.

In [ ]:
X = df['clean_text'].astype(str)
y = df['sentiment']

print("Fitur X:", X.shape)
print("Target y:", y.shape)
print("Distribusi kelas target:\n", y.value_counts(normalize=True) * 100)

### 3. Stratified Train-Test Split (80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training set : {len(X_train):,} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Testing set  : {len(X_test):,} ({len(X_test)/len(df)*100:.1f}%)")
print("\nDistribusi Train:\n", y_train.value_counts(normalize=True) * 100)
print("\nDistribusi Test:\n", y_test.value_counts(normalize=True) * 100)

### 4. TF-IDF Feature Extraction
- Fitted HANYA pada `X_train`.
- `X_test` ditransformasikan tanpa refit.
- Stopword bahasa Inggris tidak dihapus karena dataset multibahasa.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit & Transform pada data latih
X_train_tfidf = vectorizer.fit_transform(X_train)
# Transform pada data uji
X_test_tfidf = vectorizer.transform(X_test)

print("Dimensi X_train_tfidf :", X_train_tfidf.shape)
print("Dimensi X_test_tfidf  :", X_test_tfidf.shape)
print("Total Vocabulary      :", len(vectorizer.vocabulary_))

### 5. Validasi Artefak Tersimpan

In [ ]:
# Verifikasi artefak yang telah disimpan oleh src/feature_extraction.py
loaded_vec = joblib.load('../models/tfidf_vectorizer.pkl')
loaded_x_train = sparse.load_npz('../data/processed/X_train_tfidf.npz')
loaded_x_test = sparse.load_npz('../data/processed/X_test_tfidf.npz')
loaded_y_train = pd.read_csv('../data/processed/y_train.csv')
loaded_y_test = pd.read_csv('../data/processed/y_test.csv')

print("Validasi Berhasil!")
print("Loaded X_train shape:", loaded_x_train.shape)
print("Loaded X_test shape :", loaded_x_test.shape)
print("Loaded y_train shape:", loaded_y_train.shape)
print("Loaded y_test shape :", loaded_y_test.shape)